In [1]:
import pandas as pd
from pathlib import Path
import sys

# --- Configuration Constants ---
# Centralize clinical definitions and magic values for easy updates.
CONFIG = {
    "PRE_OP_WINDOW_DAYS": 90,
    "POST_OP_WINDOW_HOURS": 48,
    "MAX_PRE_OP_SCR": 4.5,
    "CREATININE_ITEM_NAME": "creatinine",
    "DIALYSIS_ITEM_NAME": "crrt",
    "EXCLUSION_PREFIXES": ["10", "0TY", "B50", "B51"],
    "EPSILON": 1e-9, # Small constant to prevent division by zero
    "REQUIRED_OP_COLS": [
        'op_id', 'subject_id', 'age', 'antype', 'asa', 'icd10_pcs',
        'height', 'weight', 'opstart_time', 'opend_time', 'department'
    ]
}

def load_and_validate_data():
    """Loads and validates the necessary CSV files."""
    try:
        # Define the base paths for the data files
        inspire_path = Path("/home/server/Projects/data/INSPIRE/physionet.org/files/inspire/1.3")
        aki_path = Path("/home/server/Projects/data/AKI/") # Path for intraop data
        
        # Define full paths for each required file
        ops_path = inspire_path / "operations.csv"
        labs_path = inspire_path / "labs.csv"
        ward_vitals_path = inspire_path / "ward_vitals.csv"
        intraop_path = aki_path / "feature_engineered.csv"

        # Load the dataframes from the specified paths
        df_operations = pd.read_csv(ops_path)
        df_labs = pd.read_csv(labs_path)
        df_ward = pd.read_csv(ward_vitals_path)
        print(f"Loading intraoperative data from: {intraop_path}")
        df_intraop = pd.read_csv(intraop_path, usecols=['op_id'])


    except FileNotFoundError as e:
        print(f"Error: {e}. Data file not found.", file=sys.stderr)
        return None, None, None, None

    if not all(col in df_operations.columns for col in CONFIG["REQUIRED_OP_COLS"]):
        print("Error: 'operations.csv' is missing required columns.", file=sys.stderr)
        return None, None, None, None

    # Convert time columns to datetime, coercing errors to NaT (which acts like NaN)
    for df, col in [(df_operations, 'opstart_time'), (df_operations, 'opend_time'), (df_labs, 'chart_time')]:
        df[col] = pd.to_datetime(df[col], errors='coerce')

    return df_operations, df_labs, df_ward, df_intraop

def process_labs(df_ops, df_labs):
    """
    Pre-processes lab data to efficiently find required creatinine values.
    """
    creatinine_labs = df_labs[df_labs['item_name'] == CONFIG["CREATININE_ITEM_NAME"]].dropna(subset=['value', 'chart_time'])
    merged_labs = pd.merge(df_ops[['op_id', 'subject_id', 'opstart_time', 'opend_time']], creatinine_labs, on='subject_id')

    pre_op_mask = (merged_labs['chart_time'] < merged_labs['opstart_time']) & \
                  (merged_labs['chart_time'] >= merged_labs['opstart_time'] - pd.Timedelta(days=CONFIG["PRE_OP_WINDOW_DAYS"]))
    last_pre_op_scr = merged_labs[pre_op_mask].sort_values('chart_time').groupby('op_id').tail(1)[['op_id', 'value']]
    last_pre_op_scr = last_pre_op_scr.rename(columns={'value': 'preop_creatinine'})

    post_op_mask = (merged_labs['chart_time'] > merged_labs['opend_time']) & \
                   (merged_labs['chart_time'] <= merged_labs['opend_time'] + pd.Timedelta(hours=CONFIG["POST_OP_WINDOW_HOURS"]))
    ops_with_postop_scr = merged_labs[post_op_mask]['op_id'].unique()
    
    post_op_2d_mask = (merged_labs['chart_time'] > merged_labs['opend_time']) & \
                      (merged_labs['chart_time'] <= merged_labs['opend_time'] + pd.Timedelta(days=2))
    post_op_7d_mask = (merged_labs['chart_time'] > merged_labs['opend_time']) & \
                      (merged_labs['chart_time'] <= merged_labs['opend_time'] + pd.Timedelta(days=7))

    max_postop_2d = merged_labs[post_op_2d_mask].groupby('op_id')['value'].max().rename('postop_creatinine_2_days')
    max_postop_7d = merged_labs[post_op_7d_mask].groupby('op_id')['value'].max().rename('postop_creatinine_7_days')
    
    processed_labs = pd.merge(last_pre_op_scr, max_postop_2d, on='op_id', how='left')
    processed_labs = pd.merge(processed_labs, max_postop_7d, on='op_id', how='left')
    
    return processed_labs, ops_with_postop_scr

def apply_filters(df_operations, processed_labs_df, ops_with_postop_scr, df_intraop):
    """Applies the sequential filtering logic to the cohort."""
    cohort_counts = []
    df_cohort = df_operations.copy()

    def record_step(description, df, reason=""):
        count = len(df['op_id'].unique())
        cohort_counts.append({"desc": description, "count": count, "reason": reason})
        if df.empty:
            print(f"Warning: Cohort empty after: '{description}'. Halting.", file=sys.stderr)
        return df.empty

    # --- Filtering Steps ---
    if record_step('Total operations recorded', df_cohort): return None, cohort_counts
    
    # --- MODIFICATION ---
    # This correctly drops rows if EITHER opstart_time OR opend_time is missing (NaT).
    # The numerical discrepancy between diagrams is due to differences in the source data
    # being read by the old vs. new scripts, not a flaw in this logic.
    df_cohort.dropna(subset=['opstart_time', 'opend_time'], inplace=True)
    if record_step('Operations after excluding unrecorded start/end time', df_cohort, "Unrecorded start or end time"): return None, cohort_counts
    
    df_cohort = df_cohort[df_cohort['age'] >= 18]
    if record_step('Adult operations (>= 18 years)', df_cohort, "Patient age < 18"): return None, cohort_counts
    
    df_cohort = df_cohort[df_cohort['asa'] < 6]
    if record_step('After excluding ASA VI (Organ Donors)', df_cohort, "ASA status VI"): return None, cohort_counts
    
    df_cohort = df_cohort[df_cohort['department'] != 'PED']
    if record_step('After excluding Pediatrics department', df_cohort, "Pediatrics department"): return None, cohort_counts

    # --- MODIFICATION START ---
    # Replaced the previous height/weight filter with a more robust two-stage check.
    # 1. First, drop rows where height or weight is missing (NaN).
    df_cohort.dropna(subset=['height', 'weight'], inplace=True)
    # 2. Then, drop rows where height or weight is zero.
    df_cohort = df_cohort[(df_cohort['height'] > 0) & (df_cohort['weight'] > 0)]
    if record_step('After excluding invalid height/weight', df_cohort, "Missing (NaN) or zero value height/weight"): return None, cohort_counts
    # --- MODIFICATION END ---
    
    df_cohort = df_cohort[df_cohort['antype'] != 'Regional']
    if record_step('Operations after excluding Regional antype', df_cohort, "Regional antype"): return None, cohort_counts

    df_cohort = df_cohort[~df_cohort['icd10_pcs'].str.startswith(tuple(CONFIG["EXCLUSION_PREFIXES"]), na=False)]
    if record_step('Operations after excluding specific prefixes', df_cohort, "Specified ICD-10-PCS prefix"): return None, cohort_counts
    
    # This step performs an inner merge to keep only operations
    # that have corresponding intraoperative data.
    df_cohort = pd.merge(df_cohort, df_intraop[['op_id']].drop_duplicates(), on='op_id', how='inner')
    if record_step('Operations after excluding missing intraoperative data', df_cohort, "No recorded intraoperative data"): return None, cohort_counts

    # Merge pre-processed lab data and apply lab-based filters
    df_cohort = pd.merge(df_cohort, processed_labs_df, on='op_id', how='inner')
    if record_step('Operations after excluding unrecorded preoperative creatinine', df_cohort, "No pre-op sCr in 90 days"): return None, cohort_counts
    
    df_cohort = df_cohort[df_cohort['op_id'].isin(ops_with_postop_scr)]
    if record_step('Operations after excluding missing creatinine within 48h post-op', df_cohort, "No post-op sCr in 48 hours"): return None, cohort_counts
    
    df_cohort = df_cohort[df_cohort['preop_creatinine'] <= CONFIG["MAX_PRE_OP_SCR"]]
    if record_step('Operations after excluding preoperative creatinine > 4.5', df_cohort, "Pre-op sCr > 4.5 mg/dL"): return None, cohort_counts

    return df_cohort, cohort_counts

def calculate_aki_stages(df_final_cohort, df_ward):
    """Calculates AKI stages for the final cohort based on KDIGO criteria."""
    df_aki = df_final_cohort.copy()
    
    df_dialysis = df_ward[df_ward['item_name'] == CONFIG["DIALYSIS_ITEM_NAME"]]
    df_dialysis = df_dialysis[['subject_id', 'value']].rename(columns={'value': 'dialysis'}).drop_duplicates('subject_id')
    df_aki = pd.merge(df_aki, df_dialysis, on='subject_id', how='left').fillna({'dialysis': 0})

    # Safely calculate 7-day creatinine ratio
    df_aki['crt_7_day_ratio'] = df_aki["postop_creatinine_7_days"] / (df_aki["preop_creatinine"] + CONFIG["EPSILON"])
    
    # --- KDIGO Stage Definitions ---
    aki_stage1 = ((df_aki['postop_creatinine_2_days'] - df_aki['preop_creatinine']) >= 0.3) | \
                 ((df_aki['crt_7_day_ratio'] >= 1.5) & (df_aki['crt_7_day_ratio'] < 2))
    aki_stage2 = (df_aki['crt_7_day_ratio'] >= 2) & (df_aki['crt_7_day_ratio'] < 3)
    aki_stage3 = (df_aki['crt_7_day_ratio'] >= 3) | (df_aki["postop_creatinine_7_days"] >= 4) | (df_aki['dialysis'] > 0)
    
    # Final boolean outcome is defined by Stage 2 or 3, as per original analysis
    df_aki['aki_boolean'] = aki_stage2 | aki_stage3
    return df_aki

def generate_dot_file(cohort_counts, filename="consort_diagram.dot"):
    """Generates a DOT file for Graphviz from the cohort counts."""
    if not cohort_counts or len(cohort_counts) < 3:
        print("Not enough data to generate a diagram.", file=sys.stderr)
        return
    dot_lines = [
        'digraph consort_flow {',
        '    graph [splines=ortho, nodesep=0.8, ranksep=0.8];',
        '    node [shape=box, style="rounded", fontname="Arial", fontsize=10];',
        '    edge [fontname="Arial", fontsize=9, arrowhead=normal];\n'
    ]
    for i in range(len(cohort_counts) - 2):
        step = cohort_counts[i]
        # Escape HTML special characters to prevent syntax errors
        safe_desc = step["desc"].replace(">", "&gt;").replace("<", "&lt;")
        label = safe_desc.replace("(", "<BR/>(")
        dot_lines.append(f'    N{i} [label=<{label}<BR/><B>(n={step["count"]:,})</B>>];')
        if i > 0:
            prev_step = cohort_counts[i-1]
            excluded_count = prev_step["count"] - step["count"]
            reason = cohort_counts[i]["reason"].replace(">", "&gt;").replace("<", "&lt;")
            dot_lines.extend([
                f'    X{i} [label=<Excluded (n={excluded_count:,})<BR ALIGN="LEFT"/>- {reason}<BR ALIGN="LEFT"/>>, shape=plaintext, fontcolor="grey40"];',
                f'    N{i-1} -> N{i};',
                f'    {{ rank=same; N{i}; X{i}; }}\n'
            ])
    final_cohort_node = f'N{len(cohort_counts) - 3}'
    
    # Escape final node descriptions as well
    neg_aki_desc = cohort_counts[-2]["desc"].replace(">", "&gt;").replace("<", "&lt;")
    pos_aki_desc = cohort_counts[-1]["desc"].replace(">", "&gt;").replace("<", "&lt;")
    neg_aki_step = cohort_counts[-2]
    pos_aki_step = cohort_counts[-1]

    dot_lines.extend([
        f'    N_neg [label=<{neg_aki_desc}<BR/><B>(n={neg_aki_step["count"]:,})</B>>];',
        f'    N_pos [label=<{pos_aki_desc}<BR/><B>(n={pos_aki_step["count"]:,})</B>>];',
        f'    {final_cohort_node} -> N_neg;',
        f'    {final_cohort_node} -> N_pos;',
        f'    {{ rank=same; N_neg; N_pos; }}\n',
        '}'
    ])
    with open(filename, "w") as f:
        f.write('\n'.join(dot_lines))
    print(f"\nSuccessfully generated DOT file: '{filename}'")

def main():
    """Main execution function to run the cohort selection pipeline."""
    df_ops, df_labs, df_ward, df_intraop = load_and_validate_data()
    
    if df_ops is None:
        return 1 # Indicate failure

    processed_labs, ops_with_postop = process_labs(df_ops, df_labs)
    
    final_cohort, counts = apply_filters(df_ops, processed_labs, ops_with_postop, df_intraop)
    
    if final_cohort is None or final_cohort.empty:
        print("Processing halted due to empty cohort.", file=sys.stderr)
        return 1

    aki_df = calculate_aki_stages(final_cohort, df_ward)
    counts.append({"desc": "Final Cohort: Negative AKI Cases", "count": (~aki_df['aki_boolean']).sum()})
    counts.append({"desc": "Final Cohort: Positive AKI Cases", "count": aki_df['aki_boolean'].sum()})
    
    print("=" * 70, "\nCOHORT SELECTION RESULTS\n", "=" * 70)
    previous_count = 0
    for step in counts:
        print(f"{step['desc']:<60} n = {step['count']:,}")
        if "Final Cohort:" not in step['desc'] and step.get("reason"):
            excluded = previous_count - step['count']
            if excluded > 0:
                print(f"  └─ Excluded: {excluded:,} ({step['reason']})")
        previous_count = step['count']
    print("=" * 70)
    
    generate_dot_file(counts)
    return 0 # Indicate success

if __name__ == '__main__':
    sys.exit(main())


Loading intraoperative data from: /home/server/Projects/data/AKI/feature_engineered.csv
COHORT SELECTION RESULTS
Total operations recorded                                    n = 130,960
Operations after excluding unrecorded start/end time         n = 130,946
  └─ Excluded: 14 (Unrecorded start or end time)
Adult operations (>= 18 years)                               n = 130,946
After excluding ASA VI (Organ Donors)                        n = 127,342
  └─ Excluded: 3,604 (ASA status VI)
After excluding Pediatrics department                        n = 127,304
  └─ Excluded: 38 (Pediatrics department)
After excluding invalid height/weight                        n = 125,539
  └─ Excluded: 1,765 (Missing (NaN) or zero value height/weight)
Operations after excluding Regional antype                   n = 125,382
  └─ Excluded: 157 (Regional antype)
Operations after excluding specific prefixes                 n = 122,548
  └─ Excluded: 2,834 (Specified ICD-10-PCS prefix)
Operations after exclu

SystemExit: 0

/home/server/Projects/VitalDB-Dimensionality-Reduction/.venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [12]:
#!/usr/bin/env python3
import os
import pandas as pd
import numpy as np

# =============================================================================
# SCRIPT CONFIGURATION
# =============================================================================
# The column name for the target variable (Acute Kidney Injury)
TARGET = 'aki_boolean'

# --- I/O Configuration ---
# Set the base directory where your data files are stored.
# IMPORTANT: You may need to change this path to match your system.
BASE_DATA_DIR = '/home/server/Projects/data/AKI/'

# --- Dataset Configurations ---
# A list of dictionaries, where each dictionary defines a dataset to analyze.
datasets_to_run = [
    {'name': 'preop', 'path': os.path.join(BASE_DATA_DIR, 'tabular_preop.csv')},
    {'name': 'intraop', 'path': os.path.join(BASE_DATA_DIR, 'tabular_intraop.csv')},
    {'name': 'combined', 'path': os.path.join(BASE_DATA_DIR, 'tabular_combined.csv')},
]

# =============================================================================
# MAIN EXECUTION BLOCK
# =============================================================================
def analyze_datasets():
    """
    Loops through the specified datasets, loads them, and prints an analysis.
    """
    print("--- Starting Data Investigation ---")

    for dataset_info in datasets_to_run:
        dataset_name = dataset_info['name']
        file_path = dataset_info['path']

        print(f"\n{'=' * 25} ANALYZING: {dataset_name.upper()} DATASET {'=' * 25}")
        print(f"File path: {file_path}")

        # --- Load Data ---
        try:
            df = pd.read_csv(file_path)
            print(f"--> Successfully loaded {dataset_name}.csv")
        except FileNotFoundError:
            print(f"--> ERROR: Data file not found. Skipping.")
            continue
        except Exception as e:
            print(f"--> ERROR: An unexpected error occurred while loading the file: {e}")
            continue

        # --- Basic Statistics ---
        if not df.empty:
            total_rows = len(df)
            
            if TARGET in df.columns:
                positive_cases = df[TARGET].sum()
                aki_rate = (positive_cases / total_rows) * 100 if total_rows > 0 else 0
                
                print("\n--- Basic Statistics ---")
                print(f"Total Rows: {total_rows}")
                print(f"Positive AKI Cases (where '{TARGET}' == 1): {positive_cases}")
                print(f"Negative AKI Cases (where '{TARGET}' == 0): {total_rows - positive_cases}")
                print(f"Overall AKI Rate: {aki_rate:.2f}%")
            else:
                print(f"\n--- WARNING: Target column '{TARGET}' not found in this dataset. ---")


            # --- Data Schema and Memory Usage ---
            print("\n--- Data Schema (Columns, Non-Null Counts, Dtypes) ---")
            # Using a buffer to capture the output of df.info() as a string
            from io import StringIO
            buffer = StringIO()
            df.info(buf=buffer)
            info_str = buffer.getvalue()
            print(info_str)

            # --- Descriptive Statistics for Numerical Columns ---
            print("\n--- Descriptive Statistics (for numerical columns) ---")
            # Include all number types and suppress scientific notation for clarity
            pd.set_option('display.float_format', lambda x: '%.3f' % x)
            print(df.describe(include=np.number))
            
        else:
            print("--> The loaded DataFrame is empty. No analysis to perform.")

    print(f"\n{'=' * 25} ANALYSIS COMPLETE {'=' * 25}")


if __name__ == "__main__":
    analyze_datasets()


--- Starting Data Investigation ---

========================= ANALYZING: PREOP DATASET =========================
File path: /home/server/Projects/data/AKI/tabular_preop.csv
--> Successfully loaded preop.csv

--- Basic Statistics ---
Total Rows: 60824
Positive AKI Cases (where 'aki_boolean' == 1): 2734
Negative AKI Cases (where 'aki_boolean' == 0): 58090
Overall AKI Rate: 4.49%

--- Data Schema (Columns, Non-Null Counts, Dtypes) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60824 entries, 0 to 60823
Data columns (total 59 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   op_id                  60824 non-null  float64
 1   age                    60824 non-null  float64
 2   sex                    60824 non-null  float64
 3   height                 60824 non-null  float64
 4   weight                 60824 non-null  float64
 5   asa                    60824 non-null  float64
 6   emop                   60824 non-